# 🎬 Auto Video Reup Studio (YouTube & Facebook Automation)
**Khởi chạy 1-Click trên Google Cloud (Google Colab Free)**
- Miễn phí 100%, không cần thẻ tín dụng, sử dụng chip và RAM mạnh mẽ từ máy chủ Google.
- Tách biệt hoàn toàn tab YouTube và Facebook.
- Tự động theo dõi kênh qua RSS, lách bản quyền FFmpeg, AI đổi tên video và sinh thumbnail mới.

In [ ]:
# @title 🚀 BƯỚC 1: Cài đặt môi trường & Khởi tạo dự án (Bấm nút Run/Play bên trái)
import os, sys, json, base64, subprocess, time, re

print('⏳ [1/3] Đang cài đặt FFmpeg, bộ font Tiếng Việt và thư viện Python trên Google Cloud...')
os.system('apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core > /dev/null 2>&1')
os.system('pip install -q fastapi uvicorn yt-dlp pillow requests feedparser apscheduler python-multipart jinja2 google-genai openai > /dev/null 2>&1')

print('⏳ [2/3] Đang tải công cụ tạo đường link công khai Cloudflare...')
if not os.path.exists('/usr/local/bin/cloudflared'):
    os.system('wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared')
    os.system('chmod +x /usr/local/bin/cloudflared')

print('⏳ [3/3] Đang giải nén mã nguồn Auto Reup Studio...')
app_dir = '/content/auto_reup'
os.makedirs(app_dir, exist_ok=True)

packed_b64 = '...'
raw_json = base64.b64decode(packed_b64.encode('utf-8')).decode('utf-8')
files_dict = json.loads(raw_json)

for rel_path, content in files_dict.items():
    full_file_path = os.path.join(app_dir, rel_path)
    os.makedirs(os.path.dirname(full_file_path), exist_ok=True)
    with open(full_file_path, 'w', encoding='utf-8') as f:
        f.write(content)

# Tạo các thư mục lưu trữ
for sub in ['storage/temp', 'storage/thumbnails', 'storage/audio', 'storage/logs', 'static/fonts']:
    os.makedirs(os.path.join(app_dir, sub), exist_ok=True)

# Copy font hệ thống tiếng Việt nếu chưa có
font_target = os.path.join(app_dir, 'static/fonts/font.ttf')
if not os.path.exists(font_target):
    os.system(f'cp /usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf {font_target} 2>/dev/null || true')

print('✅ Cài đặt hoàn tất 100%! Bắt đầu khởi chạy máy chủ...')


In [ ]:
# @title 🌐 BƯỚC 2: Khởi chạy Web & Lấy Link truy cập công khai
import subprocess, time, re, sys

os.chdir('/content/auto_reup')

# Khởi chạy Uvicorn Backend
server_proc = subprocess.Popen(
    ['uvicorn', 'app:app', '--host', '127.0.0.1', '--port', '7860'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

# Khởi chạy Cloudflare Tunnel để tạo link công khai HTTPS
tunnel_proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

public_url = None
print('Đang lấy link truy cập từ máy chủ Google Cloud...')
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if public_url:
    print('\n' + '='*70)
    print('🎉 MÁY CHỦ AUTO REUP ĐÃ HOẠT ĐỘNG THÀNH CÔNG TRÊN GOOGLE CLOUD!')
    print('👉 BẤM VÀO ĐƯỜNG LINK DƯỚI ĐÂY ĐỂ MỞ WEB TRÊN ĐIỆN THOẠI/MÁY TÍNH:')
    print(f'   🔗 {public_url}')
    print('='*70 + '\n')
    print('(Giữ nguyên tab Google Colab này mở để web hoạt động liên tục)\n')
else:
    print('⚠️ Chưa lấy được URL tunnel, đang kết nối lại...')

# Giữ tiến trình chạy để Colab không tắt
try:
    while True:
        time.sleep(10)
except KeyboardInterrupt:
    print('Dừng server.')
    server_proc.terminate()
    tunnel_proc.terminate()
